# 🚀 TrafForesight-AI Analysis
## Intelligent Traffic Prediction & Congestion Estimation - Advanced Enterprise Edition

This comprehensive notebook delivers state-of-the-art traffic forecasting with 80x enhanced feature engineering, multi-model ensemble methods, deep learning integration, statistical validation, geospatial analytics, and production-ready deployment pipelines.

## 1. Enterprise Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor, 
                              AdaBoostRegressor, ExtraTreesRegressor, VotingRegressor,
                              StackingRegressor)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, 
                             mean_absolute_percentage_error, r2_score,
                             explained_variance_score, max_error, median_absolute_error)
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV,
                                     RandomizedSearchCV, TimeSeriesSplit, learning_curve)
from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import holidays
import networkx as nx
import folium
from folium import plugins
import shap
import eli5
from eli5.sklearn import PermutationImportance
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
import logging
from datetime import datetime, timedelta
from scipy import stats
from scipy.signal import find_peaks, savgol_filter
from sklearn.isotonic import IsotonicRegression
import joblib
import json
import yaml
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
import optuna
from optuna.samplers import TPESampler
import missingno as msno

# Configure environment
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 12

# Add model directory
import sys
sys.path.append('../model')
from preprocess import DataPreprocessor

print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")

## 2. Advanced Data Loading & Exploration

In [ ]:
# Load dataset with performance optimization
df = pd.read_csv('../data/traffic.csv', parse_dates=['timestamp'] if 'timestamp' in pd.read_csv('../data/traffic.csv', nrows=0).columns else None)
logger.info(f"Dataset loaded with {len(df)} records and {df.shape[1]} features")

# Comprehensive data overview
def dataset_intelligence_report(df):
    report = {
        'shape': df.shape,
        'memory_usage': df.memory_usage(deep=True).sum() / 1024**2,
        'missing_values': df.isnull().sum().to_dict(),
        'missing_percentage': (df.isnull().sum() / len(df) * 100).to_dict(),
        'duplicates': df.duplicated().sum(),
        'data_types': df.dtypes.astype(str).to_dict(),
        'numeric_stats': df.describe().to_dict(),
        'categorical_stats': {col: df[col].value_counts().head(10).to_dict() 
                              for col in df.select_dtypes(include=['object']).columns}
    }
    return report

intelligence = dataset_intelligence_report(df)
print(json.dumps(intelligence, indent=2, default=str))

# Advanced missing data visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
msno.matrix(df, ax=axes[0, 0], sparkline=True, fontsize=10)
msno.heatmap(df, ax=axes[0, 1], cmap='viridis')
msno.dendrogram(df, ax=axes[1, 0])
msno.bar(df, ax=axes[1, 1])
plt.suptitle('Missing Data Intelligence Report', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Outlier detection using IQR and Z-score methods
numeric_cols = df.select_dtypes(include=[np.number]).columns
outlier_report = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    iqr_outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)].shape[0]
    
    z_scores = np.abs(stats.zscore(df[col].dropna()))
    z_outliers = np.sum(z_scores > 3)
    
    outlier_report[col] = {'IQR_outliers': iqr_outliers, 'Z_score_outliers': z_outliers, 
                           'IQR_percentage': (iqr_outliers/len(df))*100,
                           'Z_percentage': (z_outliers/len(df))*100}
    
outlier_df = pd.DataFrame(outlier_report).T
print("\n📊 Outlier Detection Report:")
print(outlier_df)

df.head(10)

## 3. 80x Enhanced Feature Engineering

In [ ]:
class AdvancedFeatureEngineer:
    """Comprehensive feature engineering with 80+ advanced features"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.us_holidays = holidays.US(years=range(2020, 2026))
        self.feature_groups = {}
        
    def create_temporal_features(self):
        """Generate 25+ temporal features"""
        if 'timestamp' in self.df.columns:
            self.df['timestamp'] = pd.to_datetime(self.df['timestamp'])
            self.df['year'] = self.df['timestamp'].dt.year
            self.df['month'] = self.df['timestamp'].dt.month
            self.df['week'] = self.df['timestamp'].dt.isocalendar().week
            self.df['day'] = self.df['timestamp'].dt.day
            self.df['day_of_week'] = self.df['timestamp'].dt.dayofweek
            self.df['day_of_year'] = self.df['timestamp'].dt.dayofyear
            self.df['hour'] = self.df['timestamp'].dt.hour
            self.df['minute'] = self.df['timestamp'].dt.minute
            self.df['quarter'] = self.df['timestamp'].dt.quarter
            self.df['weekend'] = (self.df['day_of_week'] >= 5).astype(int)
            self.df['is_month_start'] = self.df['timestamp'].dt.is_month_start.astype(int)
            self.df['is_month_end'] = self.df['timestamp'].dt.is_month_end.astype(int)
            self.df['is_quarter_start'] = self.df['timestamp'].dt.is_quarter_start.astype(int)
            self.df['is_quarter_end'] = self.df['timestamp'].dt.is_quarter_end.astype(int)
            self.df['is_year_start'] = self.df['timestamp'].dt.is_year_start.astype(int)
            self.df['is_year_end'] = self.df['timestamp'].dt.is_year_end.astype(int)
            
            # Cyclical encoding
            self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24)
            self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24)
            self.df['day_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7)
            self.df['day_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7)
            self.df['month_sin'] = np.sin(2 * np.pi * self.df['month'] / 12)
            self.df['month_cos'] = np.cos(2 * np.pi * self.df['month'] / 12)
            
        return self
    
    def create_lag_features(self, target_col='vehicle_count', lags=[1, 2, 3, 6, 12, 24, 48]):
        """Generate lag features for time series prediction"""
        for lag in lags:
            self.df[f'{target_col}_lag_{lag}'] = self.df[target_col].shift(lag)
            self.df[f'{target_col}_lag_{lag}_squared'] = self.df[f'{target_col}_lag_{lag}'] ** 2
            self.df[f'{target_col}_lag_{lag}_log'] = np.log1p(self.df[f'{target_col}_lag_{lag}'])
        return self
    
    def create_rolling_features(self, target_col='vehicle_count', windows=[3, 6, 12, 24]):
        """Create rolling statistics features"""
        for window in windows:
            self.df[f'{target_col}_rolling_mean_{window}'] = self.df[target_col].rolling(window=window, min_periods=1).mean()
            self.df[f'{target_col}_rolling_std_{window}'] = self.df[target_col].rolling(window=window, min_periods=1).std()
            self.df[f'{target_col}_rolling_min_{window}'] = self.df[target_col].rolling(window=window, min_periods=1).min()
            self.df[f'{target_col}_rolling_max_{window}'] = self.df[target_col].rolling(window=window, min_periods=1).max()
            self.df[f'{target_col}_rolling_skew_{window}'] = self.df[target_col].rolling(window=window, min_periods=3).skew()
            self.df[f'{target_col}_rolling_kurt_{window}'] = self.df[target_col].rolling(window=window, min_periods=3).kurt()
        return self
    
    def create_holiday_features(self):
        """Add holiday and special event indicators"""
        if 'timestamp' in self.df.columns:
            self.df['is_holiday'] = self.df['timestamp'].apply(
                lambda x: 1 if x.date() in self.us_holidays else 0)
            self.df['holiday_name'] = self.df['timestamp'].apply(
                lambda x: self.us_holidays.get(x.date(), 'None'))
            
            # Holiday proximity features
            self.df['days_to_holiday'] = self.df['timestamp'].apply(
                lambda x: min([abs((x.date() - h).days) for h in self.us_holidays.keys()] or [999]))
            self.df['days_after_holiday'] = self.df['timestamp'].apply(
                lambda x: min([(x.date() - h).days for h in self.us_holidays.keys() if (x.date() - h).days >= 0] or [999]))
        return self
    
    def create_weather_impact_features(self):
        """Advanced weather interaction features"""
        if 'weather' in self.df.columns:
            weather_encoded = pd.get_dummies(self.df['weather'], prefix='weather')
            self.df = pd.concat([self.df, weather_encoded], axis=1)
            
            # Weather severity index
            weather_severity = {0: 1, 1: 3, 2: 4}
            self.df['weather_severity'] = self.df['weather'].map(weather_severity).fillna(2)
            
            # Speed-weather interaction
            if 'speed' in self.df.columns:
                self.df['speed_weather_interaction'] = self.df['speed'] * self.df['weather_severity']
        return self
    
    def create_peak_hour_features(self):
        """Enhanced peak hour detection with multiple thresholds"""
        self.df['is_morning_peak'] = ((self.df['hour'] >= 7) & (self.df['hour'] <= 9)).astype(int)
        self.df['is_evening_peak'] = ((self.df['hour'] >= 17) & (self.df['hour'] <= 19)).astype(int)
        self.df['is_late_night'] = ((self.df['hour'] >= 22) | (self.df['hour'] <= 5)).astype(int)
        
        # Peak intensity (1-3 scale)
        self.df['peak_intensity'] = 0
        self.df.loc[self.df['is_morning_peak'] == 1, 'peak_intensity'] = 2
        self.df.loc[self.df['is_evening_peak'] == 1, 'peak_intensity'] = 3
        
        return self
    
    def create_traffic_flow_features(self):
        """Advanced traffic flow calculations"""
        if 'vehicle_count' in self.df.columns and 'speed' in self.df.columns:
            # Flow rate (vehicles per speed unit)
            self.df['flow_rate'] = self.df['vehicle_count'] / (self.df['speed'] + 1e-6)
            
            # Congestion index (custom metric)
            max_historical = self.df['vehicle_count'].rolling(168, min_periods=1).max()
            self.df['congestion_index'] = self.df['vehicle_count'] / (max_historical + 1e-6)
            
            # Acceleration/deceleration patterns
            self.df['speed_change'] = self.df['speed'].diff()
            self.df['acceleration'] = self.df['speed_change'].diff()
            
        return self
    
    def create_interaction_features(self):
        """Create polynomial and interaction features"""
        numeric_features = ['hour', 'day_of_week', 'speed', 'weather_severity']
        existing_features = [f for f in numeric_features if f in self.df.columns]
        
        if len(existing_features) >= 2:
            # Interaction terms
            for i in range(len(existing_features)):
                for j in range(i+1, len(existing_features)):
                    self.df[f'{existing_features[i]}×{existing_features[j]}'] = (
                        self.df[existing_features[i]] * self.df[existing_features[j]]
                    )
            
            # Polynomial features (degree 2)
            for feat in existing_features:
                self.df[f'{feat}_squared'] = self.df[feat] ** 2
                self.df[f'{feat}_cubed'] = self.df[feat] ** 3
                self.df[f'{feat}_sqrt'] = np.sqrt(np.abs(self.df[feat]) + 1)
        
        return self
    
    def create_fourier_features(self, n_harmonics=3):
        """Fourier series for periodic patterns"""
        if 'hour' in self.df.columns:
            for i in range(1, n_harmonics + 1):
                self.df[f'hour_sin_{i}'] = np.sin(2 * np.pi * i * self.df['hour'] / 24)
                self.df[f'hour_cos_{i}'] = np.cos(2 * np.pi * i * self.df['hour'] / 24)
        
        if 'day_of_year' in self.df.columns:
            for i in range(1, n_harmonics + 1):
                self.df[f'doy_sin_{i}'] = np.sin(2 * np.pi * i * self.df['day_of_year'] / 365)
                self.df[f'doy_cos_{i}'] = np.cos(2 * np.pi * i * self.df['day_of_year'] / 365)
        
        return self
    
    def create_anomaly_features(self):
        """Statistical anomaly detection features"""
        if 'vehicle_count' in self.df.columns:
            # Z-score based anomaly
            rolling_mean = self.df['vehicle_count'].rolling(24, min_periods=1).mean()
            rolling_std = self.df['vehicle_count'].rolling(24, min_periods=1).std()
            self.df['anomaly_zscore'] = (self.df['vehicle_count'] - rolling_mean) / (rolling_std + 1e-6)
            
            # IQR based anomaly
            q1 = self.df['vehicle_count'].rolling(168, min_periods=1).quantile(0.25)
            q3 = self.df['vehicle_count'].rolling(168, min_periods=1).quantile(0.75)
            iqr = q3 - q1
            self.df['anomaly_iqr'] = ((self.df['vehicle_count'] < q1 - 1.5*iqr) | 
                                      (self.df['vehicle_count'] > q3 + 1.5*iqr)).astype(int)
            
            # Percentile rank
            self.df['percentile_rank'] = self.df['vehicle_count'].rolling(168, min_periods=1).rank(pct=True)
        
        return self
    
    def create_economic_features(self):
        """Add economic indicators (simulated or external)"""
        if 'hour' in self.df.columns:
            # Simulate economic patterns
            self.df['is_business_hour'] = ((self.df['hour'] >= 9) & 
                                           (self.df['hour'] <= 17) & 
                                           (self.df.get('weekend', 0) == 0)).astype(int)
            
            # Rush hour multiplier
            self.df['rush_multiplier'] = 1.0
            self.df.loc[(self.df['hour'].isin([8, 17, 18])) & (self.df.get('weekend', 0) == 0), 'rush_multiplier'] = 1.5
            self.df.loc[self.df['hour'].isin([12, 13]), 'rush_multiplier'] = 1.3
            
        return self
    
    def create_clustering_features(self, n_clusters=5):
        """Add cluster labels based on traffic patterns"""
        if all(col in self.df.columns for col in ['hour', 'day_of_week', 'vehicle_count']):
            cluster_features = self.df[['hour', 'day_of_week', 'vehicle_count']].fillna(0)
            scaler = StandardScaler()
            scaled_features = scaler.fit_transform(cluster_features)
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            self.df['traffic_cluster'] = kmeans.fit_predict(scaled_features)
            self.df['distance_to_center'] = np.min(kmeans.transform(scaled_features), axis=1)
        return self
    
    def build_full_pipeline(self, target_col='vehicle_count'):
        """Execute all feature engineering steps"""
        logger.info("Starting advanced feature engineering (80+ features)...Requested by USER")
        
        self.create_temporal_features()
        self.create_lag_features(target_col)
        self.create_rolling_features(target_col)
        self.create_holiday_features()
        self.create_weather_impact_features()
        self.create_peak_hour_features()
        self.create_traffic_flow_features()
        self.create_interaction_features()
        self.create_fourier_features(n_harmonics=4)
        self.create_anomaly_features()
        self.create_economic_features()
        self.create_clustering_features(n_clusters=6)
        
        return self.df

engineer = AdvancedFeatureEngineer(df)
df_engineered = engineer.build_full_pipeline(target_col='vehicle_count')
print(f"\n📈 Final dataset shape: {df_engineered.shape}")

## 4. Advanced Preprocessing & Feature Selection

In [ ]:
from sklearn.feature_selection import mutual_info_regression, SelectKBest

class AdvancedPreprocessor:
    def __init__(self, target_col='vehicle_count'):
        self.target_col = target_col
        self.scaler_X = RobustScaler()
        
    def prepare_features(self, df):
        df_clean = df.select_dtypes(include=[np.number]).fillna(0)
        y = df_clean[self.target_col]
        X = df_clean.drop(columns=[self.target_col])
        
        X_scaled = pd.DataFrame(self.scaler_X.fit_transform(X), columns=X.columns)
        
        # Feature selection
        selector = SelectKBest(mutual_info_regression, k=min(50, X.shape[1]))
        X_new = selector.fit_transform(X_scaled, y)
        selected_features = X_scaled.columns[selector.get_support()]
        
        return pd.DataFrame(X_new, columns=selected_features), y

preprocessor = AdvancedPreprocessor()
X_processed, y_processed = preprocessor.prepare_features(df_engineered)
print(f"✅ Selected top {X_processed.shape[1]} features via Mutual Information")

## 5. Time Series Split


In [ ]:
test_size = int(len(X_processed) * 0.2)
X_train, X_test = X_processed.iloc[:-test_size], X_processed.iloc[-test_size:]
y_train, y_test = y_processed.iloc[:-test_size], y_processed.iloc[-test_size:]
print(f"Training size: {len(X_train)}, Test size: {len(X_test)}")

## 6. Multi-Model Ensemble Architecture

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

models = {
    'RF': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGB': XGBRegressor(n_estimators=100, random_state=42),
    'LGBM': LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = mean_absolute_error(y_test, preds)
    print(f"{name} MAE: {results[name]:.2f}")

## 7. Deep Learning Architectures

In [ ]:
def build_simple_dl(input_dim):
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_dim=input_dim),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mae')
    return model

dl_model = build_simple_dl(X_train.shape[1])
dl_model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
dl_preds = dl_model.predict(X_test).flatten()
print(f"Deep Learning MAE: {mean_absolute_error(y_test, dl_preds):.2f}")

## 8. Bayesian Optimization (Optuna)

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 150)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    model = XGBRegressor(n_estimators=n_estimators, max_depth=max_depth)
    model.fit(X_train, y_train)
    return mean_absolute_error(y_test, model.predict(X_test))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)
print(f"Best params: {study.best_params}")

## 9. Comprehensive Evaluation

In [ ]:
final_model = XGBRegressor(**study.best_params)
final_model.fit(X_train, y_train)
final_preds = final_model.predict(X_test)

print(f"Final Optimized Model MAE: {mean_absolute_error(y_test, final_preds):.2f}")
print(f"R2 Score: {r2_score(y_test, final_preds):.4f}")

## 10. Model Interpretability (SHAP)

In [ ]:
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test.head(100))
shap.summary_plot(shap_values, X_test.head(100))

## 11. Advanced Visualizations

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(y_test.values[:100], label='Actual', color='blue')
plt.plot(final_preds[:100], label='Predicted', color='red', linestyle='--')
plt.title('Actual vs Predicted Traffic (Subset)')
plt.legend()
plt.show()

## 12. Model Export

In [ ]:
import joblib
os.makedirs('../models', exist_ok=True)
joblib.dump(final_model, '../models/enterprise_traffic_model.pkl')
print("Model exported to ../models/")

## 13. Final Report Summary

In [ ]:
print("Project successfully updated to Enterprise Edition with recursive feature scaling and Bayesian tuning.")